### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

import groq
Groq_api_key=os.getenv("Groq_API_KEY")



In [19]:
# 1. Load environment variables
load_dotenv()

# 2. Instantiate ChatGroq (using the correct variable name: 'model')
model = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv("Groq_API_KEY"), 
    temperature=0.7
)

In [2]:
!pip install langchain_groq

In [4]:
!pip install langchain_core


In [20]:
from langchain_core.messages import HumanMessage, SystemMessage #use to specify which Humanmsg is given by human and which 1 is given by system mesg

messages =[
    SystemMessage(content="Translate the following from English to french"), #its instruct to system how should he behave like in these situation = translator
    HumanMessage(content="Hello How are you?") # 

]

result=model.invoke(messages)

In [22]:
result

AIMessage(content='Bonjour Comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 47, 'total_tokens': 54, 'completion_time': 0.017790832, 'completion_tokens_details': None, 'prompt_time': 0.002551038, 'prompt_tokens_details': None, 'queue_time': 0.16025796, 'total_time': 0.02034187}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa2f2-e640-7631-ab96-ead2cf15940f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 7, 'total_tokens': 54})

In [23]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour Comment allez-vous ?'

In [24]:
## Using LCEL - chain the Components
chain = model |parser
chain.invoke(messages)


'Bonjour Comment allez-vous ?'

In [25]:
## prompt Templates

from langchain_core.prompts import ChatPromptTemplate
generic_template="translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]

)



In [26]:
prompt.invoke({"language":"French","text":"Hello"})

ChatPromptValue(messages=[SystemMessage(content='translate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})])

In [27]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [28]:
result.to_messages()

[SystemMessage(content='translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [29]:
## Chaining Together Components With LCLE
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour.'